<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CIS3120-BMWB/blob/main/CIS3120_Class_MapFilterZip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# More on Accumulation: Map, Filter, List Comprehension, and Zip

**CIS 3120 -- Programming for Analytics**
**Reading:** Chapter 23

---

### Learning Objectives

By the end of this class you will be able to:

1. Use `map()` with a function to transform every element in a sequence
2. Use `filter()` with a Boolean function to select elements from a sequence
3. Write list comprehensions as a concise alternative to accumulator-pattern loops
4. Add conditional logic to list comprehensions (combining mapping and filtering)
5. Use `zip()` to iterate over multiple sequences in parallel
6. Choose the right tool for a given data-processing task

---
## Setup: Our Business Data

Throughout this notebook we will work with a consistent set of business data.
Run the cell below to load the sample data we will use in every section.

In [ ]:
# ----- Sample business data used throughout the notebook -----

# Product catalog
product_names = ["Laptop", "Mouse", "Keyboard", "Monitor", "Webcam",
                 "Headset", "USB Hub", "Docking Station"]
product_prices = [999.99, 24.99, 74.99, 349.99, 89.99,
                  59.99, 34.99, 189.99]
product_categories = ["Electronics", "Accessories", "Accessories",
                      "Electronics", "Electronics",
                      "Accessories", "Accessories", "Electronics"]

# Monthly revenue for the first six months of the year
months = ["January", "February", "March", "April", "May", "June"]
monthly_revenue = [45200, 38900, 52100, 49800, 61500, 58200]

# Customer information
customer_names = ["Alice Johnson", "Bob Smith", "Carol Lee",
                  "David Park", "Eva Martinez", "Frank Chen"]
customer_ages = [29, 17, 34, 20, 45, 19]
customer_emails = ["alice@baruch.cuny.edu", "bob@gmail.com",
                   "carol@baruch.cuny.edu", "david@yahoo.com",
                   "eva@baruch.cuny.edu", "frank@baruch.cuny.edu"]

# Transaction amounts (in USD)
transactions = [12.50, 250.00, 89.99, 430.00, 15.75,
                102.30, 67.00, 310.50, 8.99, 175.00]

print("Sample data loaded successfully.")
print(f"  {len(product_names)} products")
print(f"  {len(months)} months of revenue")
print(f"  {len(customer_names)} customers")
print(f"  {len(transactions)} transactions")

---
## I. Warm-Up: The Accumulator Pattern

Before we learn new tools, let us review the pattern we already know.

**The accumulator pattern** uses a loop to build up a new list one element at a time.

In [ ]:
# Accumulator pattern: convert each revenue figure to "thousands"
revenue_in_thousands = []
for rev in monthly_revenue:
    revenue_in_thousands.append(rev / 1000)

print("Revenue in thousands:", revenue_in_thousands)

This works perfectly, but it takes three lines just to express a simple idea:
*"Apply the same operation to every element."*

The rest of this class introduces more concise ways to express
common accumulator patterns.

---
## II. `map()` -- Transform Every Element

`map(function, iterable)` applies a function to **every item** in a sequence
and returns a *map object* (which you typically convert to a list).

**Think of it as:** "Do *this* to *each* item."

In [ ]:
# --- Demo 1: Convert product prices from USD to EUR ---
# Assume 1 USD = 0.92 EUR

def usd_to_eur(price):
    return round(price * 0.92, 2)

prices_eur = list(map(usd_to_eur, product_prices))
print("Prices in EUR:", prices_eur)

In [ ]:
# --- Demo 2: Same conversion using a lambda ---
# A lambda is a small anonymous (unnamed) function

prices_eur_v2 = list(map(lambda p: round(p * 0.92, 2), product_prices))
print("Prices in EUR (lambda):", prices_eur_v2)

In [ ]:
# --- Demo 3: Extract last names from full names ---

last_names = list(map(lambda name: name.split()[-1], customer_names))
print("Last names:", last_names)

### Key Points on `map()`

- The function you pass in must accept **one argument** (a single element).
- `map()` returns a *map object*, not a list. Wrap it in `list()` to see the results.
- You can use a named function or a `lambda`.
- `map()` replaces the accumulator pattern when the task is:
  *"transform each element in exactly the same way."*

### Exercise: `map()`

**Task:** Use `map()` to compute the NYC sales tax (8.875%) on each transaction amount.
The result should be a list of tax amounts, each rounded to 2 decimal places.

*Example:* a transaction of \$250.00 produces a tax of \$22.19

In [ ]:
# TODO: Replace the ... with your solution

tax_amounts = list(map(..., transactions))

print("Tax amounts:", tax_amounts)
# Expected first few values: [1.11, 22.19, 7.99, 38.16, ...]

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
tax_amounts = list(map(lambda t: round(t * 0.08875, 2), transactions))
```
</details>

**Stretch:** Use `map()` to convert every customer name to uppercase.

In [ ]:
# TODO: your code here

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
upper_names = list(map(lambda name: name.upper(), customer_names))
print(upper_names)
```
</details>

---
## III. `filter()` -- Select Elements That Pass a Test

`filter(function, iterable)` keeps only the items for which the function
returns `True`.

**Think of it as:** "Keep only the items where *this condition* holds."

In [ ]:
# --- Demo 1: Keep only transactions above $100 ---

big_transactions = list(filter(lambda t: t > 100, transactions))
print("Transactions over $100:", big_transactions)

In [ ]:
# --- Demo 2: Keep only Baruch email addresses ---

baruch_emails = list(filter(lambda e: e.endswith("@baruch.cuny.edu"),
                            customer_emails))
print("Baruch emails:", baruch_emails)

In [ ]:
# --- Demo 3: Combining map and filter ---
# Goal: take transactions over $100 and convert them to thousands

over_100 = filter(lambda t: t > 100, transactions)
in_thousands = list(map(lambda t: round(t / 1000, 3), over_100))
print("Large transactions (in thousands):", in_thousands)

### Key Points on `filter()`

- The function must return a **Boolean** (`True` or `False`).
- Like `map()`, `filter()` returns a *filter object* -- wrap in `list()`.
- `filter()` replaces the accumulator pattern when the task is:
  *"keep items that satisfy a condition."*
- You can chain `map()` and `filter()` together, though list comprehensions
  (next section) often make this easier to read.

### Exercise: `filter()`

**Task:** Use `filter()` to produce a list of customers who are 21 or older.

*Hint:* You need to work with `customer_ages`, but you also want the names.
Think about how `zip()` might help -- or for now, just filter the ages list
and we will revisit this with `zip()` later.

In [ ]:
# TODO: Filter customer_ages to keep only ages >= 21

ages_21_plus = list(filter(..., customer_ages))

print("Ages 21+:", ages_21_plus)
# Expected: [29, 34, 45]

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
ages_21_plus = list(filter(lambda age: age >= 21, customer_ages))
```
</details>

**Stretch:** Filter `product_names` to keep only those containing the substring `"USB"` or `"Dock"`.

In [ ]:
# TODO: your code here

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
filtered = list(filter(lambda name: "USB" in name or "Dock" in name,
                       product_names))
print(filtered)
```
</details>

---
## IV. List Comprehensions -- The Python "Swiss Army Knife"

A **list comprehension** is a compact syntax that can replace the accumulator
pattern, `map()`, `filter()`, or both at once.

**Basic syntax (mapping):**
```python
[expression for item in iterable]
```

**With a condition (filtering):**
```python
[expression for item in iterable if condition]
```

In [ ]:
# --- Demo 1: Comprehension as map ---
# Convert prices to EUR (same as our map example)

prices_eur_comp = [round(p * 0.92, 2) for p in product_prices]
print("Prices in EUR (comprehension):", prices_eur_comp)

In [ ]:
# --- Demo 2: Comprehension as filter ---
# Keep transactions over $100

big_trans_comp = [t for t in transactions if t > 100]
print("Over $100 (comprehension):", big_trans_comp)

In [ ]:
# --- Demo 3: Map + Filter in one expression ---
# Convert to EUR, but only for products over $50

expensive_in_eur = [round(p * 0.92, 2)
                    for p in product_prices
                    if p > 50]
print("Expensive products in EUR:", expensive_in_eur)

### Side-by-Side Comparison

Let us see the same task expressed three ways:
*"Double every transaction over \$100."*

In [ ]:
# --- Approach 1: Accumulator pattern (for loop) ---
result_loop = []
for t in transactions:
    if t > 100:
        result_loop.append(t * 2)

# --- Approach 2: map + filter ---
result_mapfilter = list(map(lambda t: t * 2,
                            filter(lambda t: t > 100, transactions)))

# --- Approach 3: List comprehension ---
result_comp = [t * 2 for t in transactions if t > 100]

print("Loop:         ", result_loop)
print("Map + Filter: ", result_mapfilter)
print("Comprehension:", result_comp)
print("All equal?    ", result_loop == result_mapfilter == result_comp)

### When to Use Which?

| Approach | Best for | Watch out for |
|---|---|---|
| `for` loop | Complex logic, multiple accumulators, side effects | More verbose |
| `map()` / `filter()` | Passing an existing named function | Harder to read when chained |
| List comprehension | Most mapping/filtering tasks | Can become unreadable if too complex |

**Rule of thumb for this course:** if a list comprehension fits on one or two
lines and is easy to read, prefer it. Otherwise, use a regular loop.

### Exercise 1: List Comprehension as Map

**Task:** Using a list comprehension, create a list of letter grades from `exam_scores`.

Use the helper function provided below.

In [ ]:
exam_scores = [92, 85, 67, 73, 55, 98, 81, 44, 77, 60]

def letter_grade(score):
    if score >= 90:
        return "A"
    elif score >= 80:
        return "B"
    elif score >= 70:
        return "C"
    elif score >= 60:
        return "D"
    else:
        return "F"

# TODO: Use a list comprehension that calls letter_grade()
grades = ...

print("Grades:", grades)
# Expected: ['A', 'B', 'D', 'C', 'F', 'A', 'B', 'F', 'C', 'D']

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
grades = [letter_grade(s) for s in exam_scores]
```
</details>

### Exercise 2: List Comprehension as Map + Filter

**Task:** From the product data, produce a list of product names
in the `"Electronics"` category with a price under \$500.

*Hint:* Use `zip()` to pair up names, prices, and categories -- or use indexing.
We have not formally covered `zip()` yet, so here is a starter using `range()`.

In [ ]:
# TODO: Complete the list comprehension
# Use indexing with range(len(...))

affordable_electronics = [product_names[i]
                          for i in range(len(product_names))
                          if ...]

print("Affordable electronics:", affordable_electronics)
# Expected: ['Monitor', 'Webcam', 'Docking Station']

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
affordable_electronics = [product_names[i]
                          for i in range(len(product_names))
                          if product_categories[i] == "Electronics"
                          and product_prices[i] < 500]
```
</details>

---
## V. `zip()` -- Iterate Over Parallel Sequences

`zip()` takes two or more sequences and **pairs up elements by position**,
producing tuples.

**Think of it as:** "Walk through these lists side by side."

```
names:  [  "A",   "B",   "C" ]
prices: [ 10.0,  20.0,  30.0 ]
           |       |       |
zip  ->  ("A",10) ("B",20) ("C",30)
```

In [ ]:
# --- Demo 1: Pair product names with prices ---

product_pairs = list(zip(product_names, product_prices))
print("Product pairs:")
for name, price in product_pairs:
    print(f"  {name}: ${price:.2f}")

In [ ]:
# --- Demo 2: Three lists -- compute weighted average ---
# Midterm is 40% of the final grade, final exam is 60%

students = ["Alice", "Bob", "Carol"]
midterm_scores = [88, 72, 95]
final_scores = [92, 80, 89]

for student, mid, final in zip(students, midterm_scores, final_scores):
    weighted = mid * 0.4 + final * 0.6
    print(f"  {student}: midterm={mid}, final={final}, weighted={weighted:.1f}")

In [ ]:
# --- Demo 3: zip inside a list comprehension ---
# Create formatted "Month: $Revenue" strings

revenue_report = [f"{m}: ${r:,.0f}" for m, r in zip(months, monthly_revenue)]
print("Revenue report:")
for line in revenue_report:
    print(f"  {line}")

### Key Points on `zip()`

- Returns a *zip object* (wrap in `list()` to inspect).
- Stops at the **shortest** sequence if lengths differ.
- Very useful combined with list comprehensions.
- Eliminates awkward `range(len(...))` indexing when working with parallel lists.

### Now Let Us Revisit That Filter Exercise

Earlier we filtered `customer_ages` but lost the names. With `zip()` we can
keep names and ages together.

In [ ]:
# Filter customers who are 21+ and keep their names

customers_21_plus = [(name, age)
                     for name, age in zip(customer_names, customer_ages)
                     if age >= 21]

print("Customers 21 and older:")
for name, age in customers_21_plus:
    print(f"  {name} (age {age})")

### Exercise: `zip()`

**Task:** Using `zip()` and a list comprehension, produce a list of
formatted strings for months where revenue exceeded \$50,000.

Each string should look like: `"March: $52,100"`

In [ ]:
# TODO: Complete the comprehension using zip

high_revenue_months = [f"{m}: ${r:,.0f}"
                       for m, r in ...
                       if ...]

print("High-revenue months:")
for line in high_revenue_months:
    print(f"  {line}")
# Expected: March: $52,100 / May: $61,500 / June: $58,200

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
high_revenue_months = [f"{m}: ${r:,.0f}"
                       for m, r in zip(months, monthly_revenue)
                       if r > 50000]
```
</details>

---
## VI. Synthesis: Choose the Right Tool

For each scenario below, decide which approach fits best and write a
one-line (or short) solution. Think about what the task *is* before coding:

- **Transforming** every element? --> `map()` or comprehension
- **Selecting** some elements? --> `filter()` or comprehension with `if`
- **Both** transforming and selecting? --> comprehension with `if`
- **Pairing** parallel data? --> `zip()` (often inside a comprehension)

### Scenario 1
> *"Create a list of product names in ALL CAPS."*

What tool would you use?

In [ ]:
# TODO: your one-liner

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
upper_products = [name.upper() for name in product_names]
# or: list(map(str.upper, product_names))
```
</details>

### Scenario 2
> *"Get all transactions that are under \$20."*

In [ ]:
# TODO: your one-liner

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
small_transactions = [t for t in transactions if t < 20]
```
</details>

### Scenario 3
> *"Build a list of dictionaries pairing each product name with its price."*

In [ ]:
# TODO: your one-liner

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
product_dicts = [{"name": n, "price": p}
                 for n, p in zip(product_names, product_prices)]
```
</details>

### Scenario 4
> *"Compute a 15% discount on every product priced over \$100, and show the
> product name with the new price."*

In [ ]:
# TODO: your solution (may need more than one line for readability)

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
discounted = [(name, round(price * 0.85, 2))
              for name, price in zip(product_names, product_prices)
              if price > 100]
for name, new_price in discounted:
    print(f"{name}: ${new_price:.2f}")
```
</details>

### Scenario 5
> *"From the customer data, build a list of email addresses for customers
> who are under 21."*

In [ ]:
# TODO: your solution

<details>
<summary><strong>Click to reveal solution</strong></summary>

```python
minor_emails = [email
                for age, email in zip(customer_ages, customer_emails)
                if age < 21]
print(minor_emails)
```
</details>

---
## VII. Wrap-Up

### What We Covered Today

| Tool | Purpose | Replaces |
|---|---|---|
| `map(fn, seq)` | Transform every element | "Apply-and-append" loop |
| `filter(fn, seq)` | Keep elements that pass a test | "If-then-append" loop |
| `[expr for x in seq]` | Map in one line | `map()` or loop |
| `[expr for x in seq if cond]` | Map + filter in one line | `map()` + `filter()` or loop |
| `zip(seq1, seq2, ...)` | Pair elements by position | `range(len(...))` indexing |

### The Big Idea

All four constructs are shortcuts for the **accumulator pattern** you
already know. They make your code shorter and more expressive -- which
matters when you are processing data for analytics.

### What to Review

- Chapter 23 (all sections)
- Practice the chapter assessment exercises
- Try rewriting old homework solutions using comprehensions where appropriate